In [1]:
!pip install statsmodels
!pip install hmmlearn 
!pip uninstall -y scipy numpy
!pip install --no-cache-dir numpy scipy


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip
Found existing installation: scipy 1.15.3
Uninstalling scipy-1.15.3:
  Successfully uninstalled scipy-1.15.3
Found existing installation: numpy 1.26.4
Not uninstalling numpy at /toolkit-cache/0.2.26/python3.10/kernel-libs/lib/python3.10/site-packages, outside environment /root/venv
Can't uninstall 'numpy'. No files were found to uninstall.
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 37.7/37.7 MB 526.5 MB/s eta 0:00:00

[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
import statsmodels.api as sm
# Load data
y = pd.read_csv("/work/y_combined_scaled_new.csv", index_col=0)
X = pd.read_csv("/work/X_combined_scaled_new.csv", index_col=0)

X.drop("DS", axis = 1)
X.drop('DPR', axis = 1)

X = X[0:1383]
y = y[0:1383]

from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=24)

In [3]:
!pip install xgboost==3.0.2


[notice] A new release of pip is available: 23.0.1 -> 25.1.1
[notice] To update, run: pip install --upgrade pip


In [4]:
import numpy as np
import pandas as pd
from hmmlearn.hmm import GaussianHMM
import statsmodels.api as sm

# Define the features explicitly
features = ["VIX", "STR", "MKT", "Jump", "RV_day", "RV_week", "RV_month", "SPX Index - Volume"]

# Dicts to store models
hmm_models = {}
har_models = {}

for n in range(2, 11):
    print(f"\n===== Fitting HMM with {n} regimes =====")
    hmm_model = GaussianHMM(n_components=n, covariance_type='full', n_iter=200, random_state=42)
    hmm_model.fit(X_train)
    hmm_models[n] = hmm_model

    print("Transition matrix:")
    print(hmm_model.transmat_)

    # Predict regimes on training set
    hidden_states = hmm_model.predict(X_train)

    har_models[n] = {}

    for regime in range(n):
        idx = (hidden_states == regime)
        X_regime = X_train[idx]
        y_regime = y_train[idx]

        if len(y_regime) == 0:
            print(f"Regime {regime} has no data, skipping HAR fit.")
            continue

        X_regime_df = pd.DataFrame(X_regime, columns=features)

        # Add constant explicitly
        X_regime_const = sm.add_constant(X_regime_df, has_constant='add')

        # Debug info
        print(f"Training regime {regime}: X shape = {X_regime_const.shape}, y length = {len(y_regime)}")

        har_model = sm.OLS(y_regime, X_regime_const).fit()
        har_models[n][regime] = har_model

        print(f"Beta coefficients for regime {regime} with {n} regimes:")
        print(har_model.params)

        
def predict_spx(X_new, n, hmm_models=hmm_models, har_models=har_models, features=features):
    """
    Predict SPX using the HMM + HAR models dictionaries.

    Parameters:
    - X_new: pd.DataFrame or np.ndarray with features matching `features`
    - n: number of regimes
    - hmm_models: dict of fitted GaussianHMM models
    - har_models: dict of HAR OLS models per regime
    - features: list of feature names

    Returns:
    - preds: np.ndarray of predictions
    """
    if not isinstance(X_new, pd.DataFrame):
        X_new = pd.DataFrame(X_new, columns=features)

    hmm_model = hmm_models[n]

    hidden_states = hmm_model.predict(X_new.values)

    preds = np.zeros(len(X_new))

    for regime in range(n):
        idx = (hidden_states == regime)
        if np.sum(idx) == 0:
            continue

        har_model = har_models[n].get(regime, None)
        if har_model is None:
            preds[idx] = np.nan
            continue

        X_regime = X_new.loc[idx, features]

        # Add constant explicitly
        X_regime_const = sm.add_constant(X_regime, has_constant='add')

        # Reorder columns exactly as training exog names
        expected_cols = har_model.model.exog_names
        X_regime_const = X_regime_const.reindex(columns=expected_cols)

        # Debug prints
        print(f"Predict regime {regime}: X shape = {X_regime_const.shape}, expected params length = {len(har_model.params)}")

        preds_here = har_model.predict(X_regime_const)

        # Check shape match before assignment
        if preds_here.shape[0] != np.sum(idx):
            raise ValueError(f"Shape mismatch: preds_here length {preds_here.shape[0]} != idx sum {np.sum(idx)}")

        preds[idx] = preds_here

    return preds


===== Fitting HMM with 2 regimes =====
Transition matrix:
[[0.33543035 0.66456965]
 [0.31824996 0.68175004]]
Training regime 0: X shape = (356, 9), y length = 356
Beta coefficients for regime 0 with 2 regimes:
const                 0.168839
VIX                   0.531877
STR                   0.085999
MKT                  -0.045644
Jump                  0.891220
RV_day                0.202548
RV_week               0.001878
RV_month             -0.053343
SPX Index - Volume    0.010096
dtype: float64
Training regime 1: X shape = (750, 9), y length = 750
Beta coefficients for regime 1 with 2 regimes:
const                 0.427615
VIX                   0.396023
STR                  -0.019642
MKT                  -0.054985
Jump                  3.686783
RV_day                0.063155
RV_week               0.124187
RV_month             -0.115219
SPX Index - Volume    0.143961
dtype: float64

===== Fitting HMM with 3 regimes =====
Model is not converging.  Current: -2858.0691816683943 is no

In [5]:
from sklearn.metrics import mean_squared_error

for n in range(2, 11):
    print(f"\n===== Fitting HMM with {n} regimes =====")
    preds = predict_spx(X_val, n)
    mse = mean_squared_error(y_val, preds)
    print(f"MSE = {mse}")


===== Fitting HMM with 2 regimes =====
Predict regime 0: X shape = (105, 9), expected params length = 9
Predict regime 1: X shape = (172, 9), expected params length = 9
MSE = 0.06834396580832029

===== Fitting HMM with 3 regimes =====
Predict regime 0: X shape = (141, 9), expected params length = 9
Predict regime 1: X shape = (79, 9), expected params length = 9
Predict regime 2: X shape = (57, 9), expected params length = 9
MSE = 0.06968359789883877

===== Fitting HMM with 4 regimes =====
Predict regime 0: X shape = (21, 9), expected params length = 9
Predict regime 1: X shape = (105, 9), expected params length = 9
Predict regime 2: X shape = (129, 9), expected params length = 9
Predict regime 3: X shape = (22, 9), expected params length = 9
MSE = 0.06147606223422575

===== Fitting HMM with 5 regimes =====
Predict regime 0: X shape = (58, 9), expected params length = 9
Predict regime 1: X shape = (97, 9), expected params length = 9
Predict regime 2: X shape = (87, 9), expected params 

In [6]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [2, 3, 4],              # Avoid deep trees to reduce overfitting in small regimes
    'learning_rate': [0.01, 0.05, 0.1],  # Conservative learning
    'subsample': [0.8, 1.0],             # Use full or nearly full sample
    'colsample_bytree': [0.8, 1.0],      # Avoid relying on few variables
    'gamma': [0, 0.1, 0.5],              # Control tree split sensitivity
    'reg_lambda': [1.0, 5.0, 10.0],      # Stronger L2 helps prevent overfit
    'reg_alpha': [0.0, 1.0, 5.0]         # L1 helps with sparsity
}

In [7]:
import numpy as np
import pandas as pd
from sklearn.model_selection import RandomizedSearchCV
from xgboost import XGBRegressor
from hmmlearn.hmm import GaussianHMM
from sklearn.linear_model import LinearRegression

# ---- CONFIG ----
n = 8  # number of regimes
features = ["VIX", "STR", "MKT", "Jump", "RV_day", "RV_week", "RV_month", "SPX Index - Volume"]
har_features = ["RV_day", "RV_week", "RV_month"]

param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [2, 3, 4],
    'learning_rate': [0.01, 0.05, 0.1],
    'subsample': [0.7, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.9, 1.0],
    'reg_lambda': [1, 5, 10]
}

# ---- MODELS ----
hmm_models = {}
har_models = {}

# ---- HMM Regime Detection ----
hmm_model = GaussianHMM(n_components=n, covariance_type='full', n_iter=200, random_state=42)
hmm_model.fit(X_train)
hmm_models[n] = hmm_model
hidden_states = hmm_model.predict(X_train)
har_models[n] = {}

# ---- Train Regime-wise HAR + XGBoost ----
for regime in range(n):
    print(f"\n[Regime {regime}] Training...")
    idx = (hidden_states == regime)
    X_regime = X_train[idx]
    y_regime = y_train[idx]

    if len(y_regime) < 5:
        print(f"Skipping regime {regime} (only {len(y_regime)} samples)")
        continue

    X_df = pd.DataFrame(X_regime, columns=features)

    # HAR model
    linreg = LinearRegression()
    linreg.fit(X_df[har_features], y_regime)
    har_pred = linreg.predict(X_df[har_features])
    resid = y_regime - har_pred

    X_df["HAR_pred"] = har_pred
    X_df["resid"] = resid

    # XGBoost tuning
    base_model = XGBRegressor(objective='reg:squarederror', random_state=42)
    search = RandomizedSearchCV(
        estimator=base_model,
        param_distributions=param_grid,
        n_iter=20,
        scoring='neg_mean_squared_error',
        cv=3,
        verbose=1,
        n_jobs=-1
    )

    print("Tuning XGBoost...")
    search.fit(X_df, y_regime)
    best_model = search.best_estimator_

    print(f"[Regime {regime}] Best Params: {search.best_params_}")

    # Save both HAR and XGBoost models
    har_models[n][regime] = {
        "har": linreg,
        "xgb": best_model
    }

# ---- Prediction Function ----
def predict_spx(X_new, n, hmm_models=hmm_models, har_models=har_models, features=features):
    if not isinstance(X_new, pd.DataFrame):
        X_new = pd.DataFrame(X_new, columns=features)

    hmm_model = hmm_models[n]
    hidden_states = hmm_model.predict(X_new.values)
    preds = np.zeros(len(X_new))

    for regime in range(n):
        idx = (hidden_states == regime)
        if np.sum(idx) == 0:
            continue

        models = har_models[n].get(regime, None)
        if models is None:
            preds[idx] = np.nan
            continue

        har_model = models["har"]
        xgb_model = models["xgb"]

        X_regime = X_new.loc[idx, :].copy()
        har_pred = har_model.predict(X_regime[har_features])
        X_regime["HAR_pred"] = har_pred
        X_regime["resid"] = 0  # can't compute true residuals out-of-sample

        preds[idx] = xgb_model.predict(X_regime)

    return preds


Model is not converging.  Current: -949.9496224317547 is not greater than -949.9421271807009. Delta is -0.007495251053796892

[Regime 0] Training...
Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[Regime 0] Best Params: {'subsample': 1.0, 'reg_lambda': 1, 'n_estimators': 300, 'max_depth': 4, 'learning_rate': 0.05, 'colsample_bytree': 0.7}

[Regime 1] Training...
Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[Regime 1] Best Params: {'subsample': 0.7, 'reg_lambda': 5, 'n_estimators': 300, 'max_depth': 2, 'learning_rate': 0.1, 'colsample_bytree': 0.7}

[Regime 2] Training...
Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[Regime 2] Best Params: {'subsample': 0.7, 'reg_lambda': 1, 'n_estimators': 200, 'max_depth': 2, 'learning_rate': 0.1, 'colsample_bytree': 0.9}

[Regime 3] Training...
Tuning XGBoost...
Fitting 3 folds for each of 20 candidates, totalling 60 fits
[Regime 3] Best Params: {'subs

In [8]:
mse = mean_squared_error(y_val, predict_spx(X_val, 3, hmm_models, har_models, features))

KeyError: 3

In [83]:
mse

0.08104933176251701

In [64]:
import joblib
# Load scaler later
scaler = joblib.load('y_scaler.pkl')
from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(scaler.inverse_transform(np.array(predict_spx(X_val, 8)).reshape(-1, 1)), scaler.inverse_transform(np.array(y_val).reshape(-1, 1)))
print(f"Mean Absolute Percentage Error (MAPE): {mape:.6f}")
print(f"MAPE as percentage: {mape * 100:.2f}%")

/root/venv/lib/python3.10/site-packages/sklearn/base.py:329: UserWarning: Trying to unpickle estimator StandardScaler from version 1.7.0 when using version 1.1.3. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


ValueError: feature_names mismatch: ['VIX', 'STR', 'MKT', 'Jump', 'DPR', 'DS', 'RV_day', 'RV_week', 'RV_month', 'HAR_pred', 'resid'] ['VIX', 'STR', 'MKT', 'Jump', 'DPR', 'DS', 'RV_day', 'RV_week', 'RV_month', 'SPX Index - Volume', 'HAR_pred', 'resid']
training data did not have the following fields: SPX Index - Volume

In [25]:
from sklearn.metrics import mean_absolute_error
mae = mean_absolute_error(predict_spx(X_test, 5), y_test)
print(mae)

Predict regime 0: X shape = (188, 10), expected params length = 10
Predict regime 1: X shape = (94, 10), expected params length = 10
Predict regime 2: X shape = (11, 10), expected params length = 10
Predict regime 4: X shape = (1, 10), expected params length = 10
0.1791983842882086


In [ ]:
import joblib
# Load scaler later
scaler = joblib.load('y_scaler.pkl')
from sklearn.metrics import mean_absolute_percentage_error
mape = mean_absolute_percentage_error(y_train.values.ravel(), y_hat_is)

import numpy as np
import statsmodels.api as sm

# Define the scoring function
def score(y_hat, y, metric="MSE"):
    if metric == "MSE":
        return np.mean((y_hat - y) ** 2)
    else:
        raise NotImplementedError(f"Metric '{metric}' not implemented.")

# Add constant to X_train for statsmodels if needed
X_train = sm.add_constant(X_train)

# Predict using your HAR model
y_pred_z = har_model.predict(X_train)  # shape: (n_samples,)

# Inverse transform to get predictions in original scale
y_hat_is = scaler.inverse_transform(np.array(y_pred_z).reshape(-1, 1))  # shape: (n_samples, 1)

# Inverse transform y_train as well
y_true_is = scaler.inverse_transform(np.array(y_train.values).reshape(-1, 1))  # shape: (n_samples, 1)

# Compute MSE
mse_is = score(y_hat_is, y_true_is, metric="MSE")

print(f"MSE: {mse_is}")


print(f"Mean Absolute Percentage Error (MAPE): {mape:.6f}")
print(f"MAPE as percentage: {mape * 100:.2f}%")